En el siguiente proyecto utilizo un dataset que contiene transacciones de un comercio de retail descargado desde Kaggle con el objetivo de analizar el comportamiento de los clientes encontrando habitos de consumo y entendiendo que tan saludable es la retencion del negocio

Preguntas guia para lograr el objetivo del proyecto:
1. Los clientes que pagan con tarjeta de crédito, ¿gastan montos más altos y variables que los que pagan en efectivo?
2. ¿Cual es el revenue por categoria?. ¿y el gasto tipico de los clientes por categoria?
3. ¿Cual es la participacion de los clientes en el revenue?
4. ¿Cuál es el nivel de retención de nuestros clientes?
5. ¿En qué horario existe la mayor cantidad de transacciones?

In [25]:
import pandas as pd
import psycopg2
from sqlalchemy import create_engine
import seaborn.objects as so
from dotenv import load_dotenv
import os

Cargo el .csv con pandas y realizo una exploracion del dataframe para saber si los datos son validos, sus formatos son correctos, que tipo de datos son, si estan completos.

In [6]:
url = 'https://raw.githubusercontent.com/MartinArielAlvarado/data-portfolio/refs/heads/main/data/Retail_Transaction_Dataset.csv'
df_transactions = pd.read_csv(url)

In [11]:
df_transactions.info()
#--------------------------------------------------------------
df_transactions.isna().sum()

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 10 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   CustomerID          100000 non-null  int64  
 1   ProductID           100000 non-null  str    
 2   Quantity            100000 non-null  int64  
 3   Price               100000 non-null  float64
 4   TransactionDate     100000 non-null  str    
 5   PaymentMethod       100000 non-null  str    
 6   StoreLocation       100000 non-null  str    
 7   ProductCategory     100000 non-null  str    
 8   DiscountApplied(%)  100000 non-null  float64
 9   TotalAmount         100000 non-null  float64
dtypes: float64(3), int64(2), str(5)
memory usage: 7.6 MB


CustomerID            0
ProductID             0
Quantity              0
Price                 0
TransactionDate       0
PaymentMethod         0
StoreLocation         0
ProductCategory       0
DiscountApplied(%)    0
TotalAmount           0
dtype: int64

In [9]:
df_transactions.head()

,CustomerID,ProductID,Quantity,Price,TransactionDate,PaymentMethod,StoreLocation,ProductCategory,DiscountApplied(%),TotalAmount
0,109318,C,7,80.079844,12/26/2023 12:32,Cash,"176 Andrew Cliffs\nBaileyfort, HI 93354",Books,18.677100,455.862764
1,993229,C,4,75.195229,8/5/2023 0:00,Cash,"11635 William Well Suite 809\nEast Kara, MT 19483",Home Decor,14.121365,258.306546
2,579675,A,8,31.528816,3/11/2024 18:51,Cash,"910 Mendez Ville Suite 909\nPort Lauraland, MO...",Books,15.943701,212.015651
3,799826,D,5,98.880218,10/27/2023 22:00,PayPal,"87522 Sharon Corners Suite 500\nLake Tammy, MO...",Books,6.686337,461.343769
4,121413,A,7,93.188512,12/22/2023 11:38,Cash,"0070 Michelle Island Suite 143\nHoland, VA 80142",Electronics,4.030096,626.030484


In [12]:
df_transactions.describe()

,CustomerID,Quantity,Price,DiscountApplied(%),TotalAmount
count,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000
mean,500463.982180,5.009290,55.067344,10.020155,248.334955
std,288460.917524,2.579808,25.971567,5.779534,184.554792
min,14.000000,1.000000,10.000430,0.000046,8.274825
25%,250693.750000,3.000000,32.549474,5.001013,95.163418
50%,499679.000000,5.000000,55.116789,10.030353,200.368393
75%,751104.750000,7.000000,77.456763,15.018367,362.009980
max,999997.000000,9.000000,99.999284,19.999585,896.141242


Notar que resulta util cambiar el tipo de dato de 'TransactionDate' de manera tal que me permita realizar operaciones para el analisis temporal. Ademas, agregar una columna 'Hour' me permite analizar una posible relacion entre horario y transacciones.

In [14]:
df_transactions['TransactionDate'] = pd.to_datetime(df_transactions['TransactionDate'])

In [16]:
df_transactions['Hour'] = df_transactions['TransactionDate'].dt.hour

Despues de la transformacion cargo el dataframe a la base de datos en PostgreSQL para estar habilitado a consultas simultaneas y no dependa de mi PC

In [26]:
load_dotenv()

connection = os.getenv('DATABASE_URL')
motor = create_engine(connection)

Task was destroyed but it is pending!
task: <Task pending name='Task-342' coro=<_async_in_context.<locals>.run_in_context() done, defined at C:\Users\alvar\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\ipykernel\utils.py:57> wait_for=<Task pending name='Task-343' coro=<Kernel.shell_main() running at C:\Users\alvar\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\ipykernel\kernelbase.py:597> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at C:\Users\alvar\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\zmq\eventloop\zmqstream.py:563]>
Exception ignored while finalizing coroutine <coroutine object Kernel.shell_main at 0x0000024F275EE790>:
Traceback (most recent call last):
  File "<string>", line 1, in <lambda>
KeyError: '__import__'
Exception ignored while finalizing coroutine <coroutine object Kernel.shell_main at 0x0000024F275EE790>:
Traceback (most recent call last):
  File "<string>", line 1, in <lambda>
KeyError: '__import_